# 02 - Leave-one-scene-out prototype sensitivity

This notebook evaluates whether the field-level spectral-support audit is sensitive to the fact that class prototypes are estimated from the same legacy SIT labels that are later audited.

For each held-out PRISMA scene, class prototypes are estimated from all other scenes and then used to score the held-out scene. The resulting field-level support scores and spectrally-atypical assignments are compared with the main global-prototype configuration.

Main outputs:

- `tables/field_spectral_support_loso_prototypes_full.csv`
- `tables/loso_prototype_sensitivity_summary_full.csv`
- `tables/scene_loso_prototype_sensitivity_full.csv`
- `tables/tab_sensitivity_loso_prototypes_full.tex`


## 1. Configuration

In [1]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

# =========================================================
# USER CONFIGURATION
# =========================================================

BASE_PATH = Path("/kaggle/input/datasets/andgu2026/prisma-data2/")

DATA_FILE = BASE_PATH / "pretraining_data.csv"
LABEL_FILE = BASE_PATH / "pretraining_data_y.csv"
FIELD_FILE = BASE_PATH / "pretraining_data_field_ids.csv"
SCENE_FILE = BASE_PATH / "pretraining_data_scene_ids.csv"

OUT_DIR = Path("/kaggle/working")
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
REPORT_DIR = OUT_DIR / "reports"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SETTING = "full"  # "full" or "summer"

SUMMER_SCENES = [
    "Scena_1", "Scena_4", "Scena_7", "Scena_8", "Scena_9",
    "Scena_10", "Scena_14", "Scena_23", "Scena_25",
]

CLASSES_TO_REMOVE = []

# If memory is limited, set this to e.g. 1000000 or 500000.
# None = use all pixels.
MAX_SAMPLES = None
SUBSAMPLE_SEED = 42

MIN_FIELD_PIXELS = 10
MIN_CLASS_PIXELS = 1000
MIN_CLASS_FIELDS = 30
MIN_CLASS_SCENES = 3

PRIORITY_SUPPORT_Q = 0.10
INTERMEDIATE_SUPPORT_Q = 0.30
PRIORITY_OUTLIER_FRACTION_Q = 0.90

CHUNK_SIZE = 250000

# If a main/global-prototype field summary already exists from the baseline notebook,
# load it instead of recomputing the global configuration.
LOAD_MAIN_FROM_EXISTING = True
MAIN_FIELD_FILE = TABLE_DIR / f"field_spectral_reliability_{SETTING}.csv"

# Optional speed controls for leave-one-scene-out prototype estimation.
# None = use all available training pixels. If set, a fixed-seed random subset of
# training pixels is used for prototype and robust-distance statistics.
LOSO_MAX_TRAIN_PIXELS_PER_CLASS = None
LOSO_RANDOM_SEED = 42

print("BASE_PATH:", BASE_PATH)
print("TABLE_DIR:", TABLE_DIR)
print("MAIN_FIELD_FILE:", MAIN_FIELD_FILE)


BASE_PATH: /kaggle/input/datasets/andgu2026/prisma-data2
TABLE_DIR: /kaggle/working/tables
MAIN_FIELD_FILE: /kaggle/working/tables/field_spectral_reliability_full.csv


## 2. Class metadata

In [2]:
CLASS_INFO = {
    111:  ("Continuous urban fabric", "artificial"),
    112:  ("Discontinuous urban fabric", "artificial"),
    121:  ("Industrial or commercial units", "artificial"),
    122:  ("Road and rail networks", "artificial"),
    123:  ("Port areas", "artificial"),
    124:  ("Airports", "artificial"),
    131:  ("Mineral extraction sites", "artificial"),
    132:  ("Dump sites", "artificial"),
    133:  ("Construction sites", "artificial"),
    141:  ("Green urban areas", "semi_dynamic"),
    142:  ("Sport and leisure facilities", "semi_dynamic"),

    2111: ("Non-irrigated arable land", "dynamic_agricultural"),
    2112: ("Non-irrigated horticultural crops", "dynamic_agricultural"),
    2121: ("Irrigated arable land", "dynamic_agricultural"),
    2123: ("Irrigated horticultural crops", "dynamic_agricultural"),

    221:  ("Vineyards", "semi_dynamic"),
    222:  ("Fruit trees and minor fruit plantations", "semi_dynamic"),
    223:  ("Olive groves", "persistent"),
    224:  ("Other permanent crops", "semi_dynamic"),
    231:  ("Dense herbaceous cover", "semi_dynamic"),
    241:  ("Annual crops associated with permanent crops", "dynamic_agricultural"),
    242:  ("Complex cultivation patterns", "dynamic_agricultural"),
    243:  ("Agricultural areas with natural vegetation", "semi_dynamic"),
    244:  ("Agro-forestry areas", "semi_dynamic"),

    311:  ("Broad-leaved forest", "persistent"),
    312:  ("Coniferous forest", "persistent"),
    313:  ("Mixed forest", "persistent"),
    314:  ("Wooded pastures and meadows", "semi_dynamic"),

    321:  ("Natural grassland, pasture, and fallow land", "semi_dynamic"),
    322:  ("Shrubland", "persistent"),
    323:  ("Sclerophyllous vegetation", "persistent"),
    324:  ("Transitional woodland/shrub", "semi_dynamic"),
    3241: ("Natural recolonization areas", "semi_dynamic"),
    3242: ("Artificial recolonization areas", "semi_dynamic"),

    331:  ("Beaches, dunes, and sand", "persistent"),
    332:  ("Bare rock and outcrops", "persistent"),
    333:  ("Sparsely vegetated areas", "semi_dynamic"),
    334:  ("Burnt areas", "semi_dynamic"),

    411:  ("Inland marshes", "semi_dynamic"),
    421:  ("Salt marshes", "semi_dynamic"),
    511:  ("Water courses", "water"),
    512:  ("Water bodies", "water"),
    521:  ("Coastal lagoons", "water"),
    522:  ("Estuaries", "water"),
    523:  ("Sea and ocean", "water"),
}

CLASS_INFO_DF = pd.DataFrame(
    [{"class_label": int(k), "class_name": v[0], "persistence_group": v[1]} for k, v in CLASS_INFO.items()]
)

display(CLASS_INFO_DF.sort_values("class_label"))


,class_label,class_name,persistence_group
0,111,Continuous urban fabric,artificial
1,112,Discontinuous urban fabric,artificial
2,121,Industrial or commercial units,artificial
3,122,Road and rail networks,artificial
4,123,Port areas,artificial
5,124,Airports,artificial
6,131,Mineral extraction sites,artificial
7,132,Dump sites,artificial
8,133,Construction sites,artificial
9,141,Green urban areas,semi_dynamic


## 3. Utility functions

In [4]:
def read_vector_csv(path, dtype=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    v = pd.read_csv(path, header=None).iloc[:, 0].values
    if dtype is not None:
        v = v.astype(dtype)
    return v


def normalize_scene_id_array(scene_ids):
    raw = pd.Series(scene_ids).astype(str).str.strip()
    cleaned = raw.str.replace(r"[-\s]+", "_", regex=True)
    extracted = cleaned.str.extract(r"(?i)scena_?(\d+)", expand=False)
    out = cleaned.copy()
    mask = extracted.notna()
    out.loc[mask] = extracted.loc[mask].astype(int).map(lambda n: f"Scena_{n}")
    return out.values


def subsample_balanced_by_scene_indices(scene_ids, max_samples=None, seed=42):
    scene_ids = np.asarray(scene_ids)
    n_total = len(scene_ids)

    if max_samples is None or n_total <= max_samples:
        return np.arange(n_total, dtype=np.int64)

    rng = np.random.default_rng(seed)
    scenes = sorted(pd.Series(scene_ids).unique())
    base_per_scene = max_samples // len(scenes)
    remainder = max_samples % len(scenes)

    chosen = []
    for i, scene in enumerate(scenes):
        idx = np.where(scene_ids == scene)[0]
        n = base_per_scene + (1 if i < remainder else 0)
        n = min(n, len(idx))
        chosen.extend(rng.choice(idx, size=n, replace=False).tolist())

    chosen = np.array(sorted(set(chosen)), dtype=np.int64)

    if len(chosen) < max_samples:
        selected_mask = np.zeros(n_total, dtype=bool)
        selected_mask[chosen] = True
        remaining = np.where(~selected_mask)[0]
        n_extra = min(max_samples - len(chosen), len(remaining))
        if n_extra > 0:
            extra = rng.choice(remaining, size=n_extra, replace=False)
            chosen = np.array(sorted(np.concatenate([chosen, extra])), dtype=np.int64)

    return chosen


def maybe_subsample_indices(idx, max_n=None, seed=42):
    idx = np.asarray(idx, dtype=np.int64)
    if max_n is None or len(idx) <= max_n:
        return idx
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(idx, size=max_n, replace=False).astype(np.int64))


def l2_normalize_rows(A, eps=1e-12):
    A = np.asarray(A, dtype=np.float32)
    norm = np.linalg.norm(A, axis=1, keepdims=True).astype(np.float32)
    return A / np.maximum(norm, eps)


def robust_median_iqr(x):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan
    q25 = np.quantile(x, 0.25)
    q50 = np.quantile(x, 0.50)
    q75 = np.quantile(x, 0.75)
    return float(q25), float(q50), float(q75), float(q75 - q25)


def spectral_scores_to_proto(X_class, proto, chunk_size=250000):
    n = X_class.shape[0]
    sad = np.empty(n, dtype=np.float32)
    edist = np.empty(n, dtype=np.float32)

    proto = proto.astype(np.float32)
    proto_norm = max(float(np.linalg.norm(proto)), 1e-12)
    proto = proto / proto_norm

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        Xc = X_class[start:end].astype(np.float32, copy=False)
        Xn = l2_normalize_rows(Xc)

        dots = np.sum(Xn * proto[None, :], axis=1)
        dots = np.clip(dots, -1.0, 1.0)
        sad[start:end] = np.arccos(dots).astype(np.float32)
        edist[start:end] = np.linalg.norm(Xn - proto[None, :], axis=1).astype(np.float32)

    return sad, edist


def estimate_median_proto(X_train):
    Xn = l2_normalize_rows(X_train)
    proto = np.median(Xn, axis=0).astype(np.float32)
    proto = proto / max(float(np.linalg.norm(proto)), 1e-12)
    return proto


def support_from_distances(sad, edist, sad_med, sad_iqr, ed_med, ed_iqr, eps=1e-12):
    sad_z = (sad - sad_med) / max(sad_iqr, eps)
    ed_z = (edist - ed_med) / max(ed_iqr, eps)
    z = np.maximum(0, np.maximum(sad_z, ed_z))
    return np.exp(-z).astype(np.float32), sad_z.astype(np.float32), ed_z.astype(np.float32)


def aggregate_field_scores(pixel_scores, field_base):
    field_scores = (
        pixel_scores.groupby(["scene_id", "field_id", "class_label"], dropna=False)
        .agg(
            spectral_support_mean=("spectral_support", "mean"),
            spectral_support_median=("spectral_support", "median"),
            spectral_support_q25=("spectral_support", lambda x: x.quantile(0.25)),
            spectral_support_q75=("spectral_support", lambda x: x.quantile(0.75)),
            sad_median=("sad", "median"),
            sad_mean=("sad", "mean"),
            sad_q25=("sad", lambda x: x.quantile(0.25)),
            sad_q75=("sad", lambda x: x.quantile(0.75)),
            edist_median=("edist", "median"),
            edist_mean=("edist", "mean"),
            outlier_fraction=("is_outlier", "mean"),
        )
        .reset_index()
    )

    out = field_base.merge(
        field_scores,
        on=["scene_id", "field_id", "class_label"],
        how="left",
    )

    # Backward-compatible aliases used by the original baseline notebook.
    out["reliability_mean"] = out["spectral_support_mean"]
    out["reliability_median"] = out["spectral_support_median"]
    out["reliability_q25"] = out["spectral_support_q25"]
    out["reliability_q75"] = out["spectral_support_q75"]
    return out


def assign_spectral_support_strata(field_summary, s_col="spectral_support_median", o_col="outlier_fraction"):
    out = field_summary.copy()
    out["audit_flag"] = "unassigned"

    out.loc[out["n_pixels"] < MIN_FIELD_PIXELS, "audit_flag"] = "too_small"
    out.loc[
        (out["n_pixels"] >= MIN_FIELD_PIXELS) & (~out["class_supported"]),
        "audit_flag"
    ] = "insufficient_class_support"

    eligible = (
        (out["n_pixels"] >= MIN_FIELD_PIXELS) &
        (out["class_supported"]) &
        out[s_col].notna() &
        out[o_col].notna()
    )

    eligible_scores = out.loc[eligible].copy()
    if len(eligible_scores) == 0:
        raise ValueError("No eligible field-scene units available for stratum assignment.")

    tau_low = eligible_scores[s_col].quantile(PRIORITY_SUPPORT_Q)
    tau_mid = eligible_scores[s_col].quantile(INTERMEDIATE_SUPPORT_Q)
    omega = eligible_scores[o_col].quantile(PRIORITY_OUTLIER_FRACTION_Q)

    is_sa = eligible & ((out[s_col] <= tau_low) | (out[o_col] >= omega))
    is_is = eligible & (out["audit_flag"] == "unassigned") & (out[s_col] <= tau_mid)
    is_st = eligible & (out["audit_flag"] == "unassigned")

    out.loc[is_sa, "audit_flag"] = "spectrally_atypical"
    out.loc[is_is, "audit_flag"] = "intermediate_support"
    out.loc[is_st, "audit_flag"] = "spectrally_typical"

    thresholds = {
        "tau_low": float(tau_low),
        "tau_mid": float(tau_mid),
        "omega": float(omega),
        "q_low": PRIORITY_SUPPORT_Q,
        "q_mid": INTERMEDIATE_SUPPORT_Q,
        "q_out": PRIORITY_OUTLIER_FRACTION_Q,
    }
    return out, thresholds


def normalize_audit_flag_names(s):
    return (
        pd.Series(s)
        .replace({
            "priority_for_inspection": "spectrally_atypical",
            "uncertain": "intermediate_support",
            "reliable": "spectrally_typical",
        })
        .values
    )


def field_keys(df, mask):
    return set(df.loc[mask, ["scene_id", "field_id", "class_label"]].apply(tuple, axis=1))


def jaccard(a, b):
    a = set(a)
    b = set(b)
    if len(a | b) == 0:
        return np.nan
    return len(a & b) / len(a | b)

print("Utility functions ready.")


Utility functions ready.


## 4. Load and filter dataset

In [5]:
for p in [DATA_FILE, LABEL_FILE, FIELD_FILE, SCENE_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Missing input file: {p}")

print("Loading X:", DATA_FILE)
X = pd.read_csv(DATA_FILE, header=None).values.astype(np.float32)

print("Loading y:", LABEL_FILE)
y = read_vector_csv(LABEL_FILE, dtype=np.int64)

print("Loading field ids:", FIELD_FILE)
field_ids = read_vector_csv(FIELD_FILE, dtype=np.int64)

print("Loading scene ids:", SCENE_FILE)
scene_ids = normalize_scene_id_array(read_vector_csv(SCENE_FILE))

if len(X) != len(y) or len(X) != len(field_ids) or len(X) != len(scene_ids):
    raise ValueError(f"Length mismatch: X={len(X)}, y={len(y)}, field_ids={len(field_ids)}, scene_ids={len(scene_ids)}")

keep = np.ones(len(y), dtype=bool)

if len(CLASSES_TO_REMOVE) > 0:
    keep &= ~np.isin(y, np.array(CLASSES_TO_REMOVE, dtype=np.int64))

if SETTING == "summer":
    keep &= np.isin(scene_ids, np.array(SUMMER_SCENES, dtype=str))
elif SETTING == "full":
    pass
else:
    raise ValueError(f"Unknown SETTING={SETTING}")

X = X[keep]
y = y[keep]
field_ids = field_ids[keep]
scene_ids = scene_ids[keep]

idx_sub = subsample_balanced_by_scene_indices(scene_ids, max_samples=MAX_SAMPLES, seed=SUBSAMPLE_SEED)
X = X[idx_sub]
y = y[idx_sub]
field_ids = field_ids[idx_sub]
scene_ids = scene_ids[idx_sub]

sample_index = np.arange(len(y), dtype=np.int64)

print("Rows after filtering/subsampling:", len(y))
print("X shape:", X.shape)
print("X memory GB:", X.nbytes / 1e9)
print("Scenes:", sorted(pd.Series(scene_ids).unique()))
print("Classes:", sorted(pd.Series(y).unique()))
print("Field-scene units:", pd.DataFrame({"scene_id": scene_ids, "field_id": field_ids}).drop_duplicates().shape[0])

gc.collect()


Loading X: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data.csv
Loading y: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_y.csv
Loading field ids: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_field_ids.csv
Loading scene ids: /kaggle/input/datasets/andgu2026/prisma-data2/pretraining_data_scene_ids.csv
Rows after filtering/subsampling: 2500000
X shape: (2500000, 191)
X memory GB: 1.91
Scenes: ['Scena_1', 'Scena_10', 'Scena_11', 'Scena_12', 'Scena_13', 'Scena_14', 'Scena_15', 'Scena_16', 'Scena_17', 'Scena_18', 'Scena_19', 'Scena_2', 'Scena_20', 'Scena_21', 'Scena_22', 'Scena_23', 'Scena_24', 'Scena_25', 'Scena_3', 'Scena_4', 'Scena_5', 'Scena_6', 'Scena_7', 'Scena_8', 'Scena_9']
Classes: [np.int64(221), np.int64(222), np.int64(223), np.int64(224), np.int64(231), np.int64(241), np.int64(242), np.int64(243), np.int64(244), np.int64(311), np.int64(312), np.int64(313), np.int64(314), np.int64(321), np.int64(322), np.int64(323), np.int64(331), n

47

## 5. Metadata and class support

In [6]:
df = pd.DataFrame({
    "sample_index": sample_index,
    "class_label": y.astype(np.int64),
    "field_id": field_ids.astype(np.int64),
    "scene_id": scene_ids.astype(str),
})
df["setting"] = SETTING

df = df.merge(CLASS_INFO_DF, on="class_label", how="left")
if df["class_name"].isna().any():
    missing = sorted(df.loc[df["class_name"].isna(), "class_label"].unique())
    print(f"WARNING: Missing CLASS_INFO for classes: {missing}")
    df["class_name"] = df["class_name"].fillna(df["class_label"].apply(lambda x: f"SIT class {int(x)}"))
    df["persistence_group"] = df["persistence_group"].fillna("unknown")

field_base = (
    df.groupby(["scene_id", "field_id", "class_label", "class_name", "persistence_group", "setting"], dropna=False)
    .agg(n_pixels=("sample_index", "size"))
    .reset_index()
)

field_support_tmp = (
    df.groupby(["scene_id", "field_id", "class_label"], dropna=False)
    .size()
    .reset_index(name="field_n_pixels")
)

class_support = (
    field_support_tmp
    .groupby("class_label", dropna=False)
    .agg(
        class_n_pixels=("field_n_pixels", "sum"),
        class_n_fields=("field_id", "count"),
        class_n_scenes=("scene_id", "nunique"),
    )
    .reset_index()
)

class_support["class_supported"] = (
    (class_support["class_n_pixels"] >= MIN_CLASS_PIXELS) &
    (class_support["class_n_fields"] >= MIN_CLASS_FIELDS) &
    (class_support["class_n_scenes"] >= MIN_CLASS_SCENES)
)

class_support = class_support.merge(CLASS_INFO_DF, on="class_label", how="left")
class_support["class_name"] = class_support["class_name"].fillna(class_support["class_label"].apply(lambda x: f"SIT class {int(x)}"))
class_support["persistence_group"] = class_support["persistence_group"].fillna("unknown")

df = df.merge(
    class_support[["class_label", "class_n_pixels", "class_n_fields", "class_n_scenes", "class_supported"]],
    on="class_label",
    how="left",
)
df["class_supported"] = df["class_supported"].fillna(False).astype(bool)

field_base = field_base.merge(
    class_support[["class_label", "class_n_pixels", "class_n_fields", "class_n_scenes", "class_supported"]],
    on="class_label",
    how="left",
)
field_base["class_supported"] = field_base["class_supported"].fillna(False).astype(bool)

supported_classes = sorted(class_support.loc[class_support["class_supported"], "class_label"].unique())
all_scenes = sorted(pd.Series(scene_ids).unique())

print("Supported classes:", supported_classes)
print("Unsupported classes:", sorted(class_support.loc[~class_support["class_supported"], "class_label"].unique()))
display(class_support.sort_values(["class_supported", "class_n_pixels"], ascending=[True, False]))


Supported classes: [np.int64(221), np.int64(222), np.int64(223), np.int64(231), np.int64(241), np.int64(242), np.int64(243), np.int64(311), np.int64(312), np.int64(313), np.int64(314), np.int64(321), np.int64(322), np.int64(323), np.int64(331), np.int64(332), np.int64(333), np.int64(2111), np.int64(2112), np.int64(2121), np.int64(3241), np.int64(3242)]
Unsupported classes: [np.int64(224), np.int64(244), np.int64(334), np.int64(2123)]


,class_label,class_n_pixels,class_n_fields,class_n_scenes,class_supported,class_name,persistence_group
3,224,828,109,10,False,Other permanent crops,semi_dynamic
23,2123,610,161,21,False,Irrigated horticultural crops,dynamic_agricultural
8,244,257,60,3,False,Agro-forestry areas,semi_dynamic
19,334,154,25,4,False,Burnt areas,semi_dynamic
2,223,734113,44931,25,True,Olive groves,persistent
20,2111,675874,55696,25,True,Non-irrigated arable land,dynamic_agricultural
0,221,244804,27419,24,True,Vineyards,semi_dynamic
22,2121,218679,3008,16,True,Irrigated arable land,dynamic_agricultural
13,321,142643,18135,25,True,"Natural grassland, pasture, and fallow land",semi_dynamic
9,311,138395,4471,25,True,Broad-leaved forest,persistent


## 6. Main/global-prototype field summary

If the baseline field summary already exists, it is loaded. Otherwise, it is recomputed using global class prototypes.

In [7]:
def compute_global_field_summary():
    score_parts = []

    for class_label in supported_classes:
        print(f"Global prototypes: class {class_label}...")
        idx = np.where((y == class_label) & df["class_supported"].values)[0]
        if len(idx) == 0:
            continue

        Xc = X[idx].astype(np.float32, copy=False)
        proto = estimate_median_proto(Xc)

        sad, edist = spectral_scores_to_proto(Xc, proto, chunk_size=CHUNK_SIZE)
        sad_q25, sad_med, sad_q75, sad_iqr = robust_median_iqr(sad)
        ed_q25, ed_med, ed_q75, ed_iqr = robust_median_iqr(edist)

        support, sad_z, ed_z = support_from_distances(sad, edist, sad_med, sad_iqr, ed_med, ed_iqr)

        sad_thr = sad_q75 + 1.5 * sad_iqr
        ed_thr = ed_q75 + 1.5 * ed_iqr
        is_outlier = (sad > sad_thr) | (edist > ed_thr)

        part = pd.DataFrame({
            "sample_index": sample_index[idx],
            "scene_id": scene_ids[idx],
            "field_id": field_ids[idx],
            "class_label": y[idx],
            "sad": sad.astype(np.float32),
            "edist": edist.astype(np.float32),
            "sad_robust_z": sad_z.astype(np.float32),
            "edist_robust_z": ed_z.astype(np.float32),
            "spectral_support": support.astype(np.float32),
            "is_outlier": is_outlier.astype(bool),
        })
        score_parts.append(part)

        del Xc, sad, edist, support, sad_z, ed_z, is_outlier, part
        gc.collect()

    pixel_scores = pd.concat(score_parts, ignore_index=True)
    global_fields = aggregate_field_scores(pixel_scores, field_base)
    global_fields, global_thresholds = assign_spectral_support_strata(global_fields)
    return global_fields, global_thresholds

if LOAD_MAIN_FROM_EXISTING and MAIN_FIELD_FILE.exists():
    print("Loading existing main field summary:", MAIN_FIELD_FILE)
    field_main = pd.read_csv(MAIN_FIELD_FILE)

    # Normalize old names to the new spectral-support terminology.
    if "spectral_support_median" not in field_main.columns and "reliability_median" in field_main.columns:
        field_main["spectral_support_median"] = field_main["reliability_median"]
        field_main["spectral_support_mean"] = field_main.get("reliability_mean", np.nan)
        field_main["spectral_support_q25"] = field_main.get("reliability_q25", np.nan)
        field_main["spectral_support_q75"] = field_main.get("reliability_q75", np.nan)

    if "audit_flag" in field_main.columns:
        field_main["audit_flag"] = normalize_audit_flag_names(field_main["audit_flag"])
    else:
        field_main, main_thresholds = assign_spectral_support_strata(field_main)

    # Ensure class support is available and consistent.
    if "class_supported" not in field_main.columns:
        field_main = field_main.merge(
            class_support[["class_label", "class_supported"]], on="class_label", how="left"
        )
        field_main["class_supported"] = field_main["class_supported"].fillna(False).astype(bool)

    # If the loaded file still uses old flag names, this has already normalized them.
    main_thresholds = None
else:
    print("Main field summary not found or LOAD_MAIN_FROM_EXISTING=False. Recomputing global configuration.")
    field_main, main_thresholds = compute_global_field_summary()

print("field_main:", field_main.shape)
print(field_main["audit_flag"].value_counts(dropna=False))
display(field_main.head())


Main field summary not found or LOAD_MAIN_FROM_EXISTING=False. Recomputing global configuration.
Global prototypes: class 221...
Global prototypes: class 222...
Global prototypes: class 223...
Global prototypes: class 231...
Global prototypes: class 241...
Global prototypes: class 242...
Global prototypes: class 243...
Global prototypes: class 311...
Global prototypes: class 312...
Global prototypes: class 313...
Global prototypes: class 314...
Global prototypes: class 321...
Global prototypes: class 322...
Global prototypes: class 323...
Global prototypes: class 331...
Global prototypes: class 332...
Global prototypes: class 333...
Global prototypes: class 2111...
Global prototypes: class 2112...
Global prototypes: class 2121...
Global prototypes: class 3241...
Global prototypes: class 3242...
field_main: (190914, 27)
audit_flag
too_small                     150076
spectrally_typical             40800
insufficient_class_support        38
Name: count, dtype: int64


,scene_id,field_id,class_label,class_name,persistence_group,setting,n_pixels,class_n_pixels,class_n_fields,class_n_scenes,class_supported,spectral_support_mean,spectral_support_median,spectral_support_q25,spectral_support_q75,sad_median,sad_mean,sad_q25,sad_q75,edist_median,edist_mean,outlier_fraction,reliability_mean,reliability_median,reliability_q25,reliability_q75,audit_flag
0,Scena_1,1,221,Vineyards,semi_dynamic,full,3,244804,27419,24,True,1.000000,1.000000,1.000000,1.000000,0.121537,0.119158,0.117686,0.121820,0.121462,0.119087,0.0,1.000000,1.000000,1.000000,1.000000,too_small
1,Scena_1,2,221,Vineyards,semi_dynamic,full,2,244804,27419,24,True,0.845717,0.845717,0.768576,0.922859,0.121937,0.121937,0.089304,0.154569,0.121796,0.121796,0.0,0.845717,0.845717,0.768576,0.922859,too_small
2,Scena_1,5,221,Vineyards,semi_dynamic,full,1,244804,27419,24,True,1.000000,1.000000,1.000000,1.000000,0.097143,0.097143,0.097143,0.097143,0.097106,0.097106,0.0,1.000000,1.000000,1.000000,1.000000,too_small
3,Scena_1,7,221,Vineyards,semi_dynamic,full,1,244804,27419,24,True,1.000000,1.000000,1.000000,1.000000,0.086871,0.086871,0.086871,0.086871,0.086844,0.086844,0.0,1.000000,1.000000,1.000000,1.000000,too_small
4,Scena_1,8,221,Vineyards,semi_dynamic,full,1,244804,27419,24,True,0.975407,0.975407,0.975407,0.975407,0.149283,0.149283,0.149283,0.149283,0.149144,0.149144,0.0,0.975407,0.975407,0.975407,0.975407,too_small


In [8]:
# =========================================================
# RECOMPUTE MAIN SPECTRAL-SUPPORT STRATA
# =========================================================

score_col = "spectral_support_median"
out_col = "outlier_fraction"

eligible_main = (
    field_main["class_supported"].eq(True)
    & (field_main["n_pixels"] >= MIN_FIELD_PIXELS)
    & field_main[score_col].notna()
    & field_main[out_col].notna()
)

tau10_main = field_main.loc[eligible_main, score_col].quantile(0.10)
tau30_main = field_main.loc[eligible_main, score_col].quantile(0.30)
omega90_main = field_main.loc[eligible_main, out_col].quantile(0.90)

print("Eligible main:", int(eligible_main.sum()))
print("tau10_main:", tau10_main)
print("tau30_main:", tau30_main)
print("omega90_main:", omega90_main)

field_main["audit_flag_recomputed"] = field_main["audit_flag"].copy()

field_main.loc[eligible_main, "audit_flag_recomputed"] = "spectrally_typical"

field_main.loc[
    eligible_main
    & (field_main[score_col] > tau10_main)
    & (field_main[score_col] <= tau30_main)
    & (field_main[out_col] < omega90_main),
    "audit_flag_recomputed"
] = "intermediate_support"

field_main.loc[
    eligible_main
    & (
        (field_main[score_col] <= tau10_main)
        | (field_main[out_col] >= omega90_main)
    ),
    "audit_flag_recomputed"
] = "spectrally_atypical"

print(field_main["audit_flag_recomputed"].value_counts(dropna=False))

Eligible main: 40800
tau10_main: 0.34423111081123353
tau30_main: 0.7282259464263916
omega90_main: 0.07142857142857142
audit_flag_recomputed
too_small                     150076
spectrally_typical             27863
intermediate_support            7132
spectrally_atypical             5805
insufficient_class_support        38
Name: count, dtype: int64


In [9]:
## Representative field-level audit examples for manuscript table
print(field_main.shape)
print(field_main.columns.tolist())

print("\nAudit flags:")
print(field_main["audit_flag_recomputed"].value_counts(dropna=False)
      if "audit_flag_recomputed" in field_main.columns
      else field_main["audit_flag"].value_counts(dropna=False))

(190914, 28)
['scene_id', 'field_id', 'class_label', 'class_name', 'persistence_group', 'setting', 'n_pixels', 'class_n_pixels', 'class_n_fields', 'class_n_scenes', 'class_supported', 'spectral_support_mean', 'spectral_support_median', 'spectral_support_q25', 'spectral_support_q75', 'sad_median', 'sad_mean', 'sad_q25', 'sad_q75', 'edist_median', 'edist_mean', 'outlier_fraction', 'reliability_mean', 'reliability_median', 'reliability_q25', 'reliability_q75', 'audit_flag', 'audit_flag_recomputed']

Audit flags:
audit_flag_recomputed
too_small                     150076
spectrally_typical             27863
intermediate_support            7132
spectrally_atypical             5805
insufficient_class_support        38
Name: count, dtype: int64


In [10]:
# =========================================================
# Representative field-level audit examples
# =========================================================

score_col = "spectral_support_median"
out_col = "outlier_fraction"
flag_col = "audit_flag_recomputed" if "audit_flag_recomputed" in field_main.columns else "audit_flag"

eligible_examples = field_main[
    field_main["class_supported"].eq(True)
    & (field_main["n_pixels"] >= MIN_FIELD_PIXELS)
    & field_main[score_col].notna()
    & field_main[out_col].notna()
    & field_main[flag_col].isin([
        "spectrally_atypical",
        "intermediate_support",
        "spectrally_typical"
    ])
].copy()

print("Eligible examples:", len(eligible_examples))
print(eligible_examples[flag_col].value_counts())


display_cols = [
    "scene_id",
    "field_id",
    "class_label",
    "class_name",
    "n_pixels",
    score_col,
    out_col,
    flag_col
]

for flag in ["spectrally_atypical", "intermediate_support", "spectrally_typical"]:
    print("\n" + "="*80)
    print(flag)
    print("="*80)
    
    tmp = eligible_examples[eligible_examples[flag_col].eq(flag)].copy()
    
    if flag == "spectrally_atypical":
        tmp = tmp.sort_values([score_col, out_col], ascending=[True, False])
    elif flag == "intermediate_support":
        tmp = tmp.sort_values(score_col, ascending=True)
    else:
        tmp = tmp.sort_values(score_col, ascending=False)
    
    display(tmp[display_cols].head(20))

Eligible examples: 40800
audit_flag_recomputed
spectrally_typical      27863
intermediate_support     7132
spectrally_atypical      5805
Name: count, dtype: int64

spectrally_atypical


,scene_id,field_id,class_label,class_name,n_pixels,spectral_support_median,outlier_fraction,audit_flag_recomputed
92782,Scena_19,10621,241,Annual crops associated with permanent crops,10,0.004542,0.900000,spectrally_atypical
107281,Scena_21,682,241,Annual crops associated with permanent crops,15,0.004777,1.000000,spectrally_atypical
161183,Scena_6,1582,241,Annual crops associated with permanent crops,10,0.005244,1.000000,spectrally_atypical
162175,Scena_6,3385,241,Annual crops associated with permanent crops,18,0.007486,1.000000,spectrally_atypical
102503,Scena_20,6764,241,Annual crops associated with permanent crops,20,0.008037,0.900000,spectrally_atypical
27361,Scena_11,9878,241,Annual crops associated with permanent crops,13,0.008363,1.000000,spectrally_atypical
107647,Scena_21,1218,241,Annual crops associated with permanent crops,22,0.008541,1.000000,spectrally_atypical
189220,Scena_9,8879,241,Annual crops associated with permanent crops,10,0.010135,1.000000,spectrally_atypical
96843,Scena_2,5971,241,Annual crops associated with permanent crops,19,0.010675,1.000000,spectrally_atypical
105243,Scena_20,11194,241,Annual crops associated with permanent crops,15,0.013468,1.000000,spectrally_atypical



intermediate_support


,scene_id,field_id,class_label,class_name,n_pixels,spectral_support_median,outlier_fraction,audit_flag_recomputed
115537,Scena_22,2183,2121,Irrigated arable land,79,0.344264,0.000000,intermediate_support
190842,Scena_9,11120,323,Sclerophyllous vegetation,93,0.344285,0.000000,intermediate_support
114667,Scena_22,688,2111,Non-irrigated arable land,10,0.344429,0.000000,intermediate_support
41349,Scena_12,14707,321,"Natural grassland, pasture, and fallow land",10,0.344505,0.000000,intermediate_support
128501,Scena_24,4798,223,Olive groves,19,0.344507,0.000000,intermediate_support
99188,Scena_20,1486,2111,Non-irrigated arable land,10,0.344632,0.000000,intermediate_support
95297,Scena_2,2826,223,Olive groves,12,0.344698,0.000000,intermediate_support
98644,Scena_20,602,223,Olive groves,19,0.344735,0.000000,intermediate_support
93478,Scena_19,11705,323,Sclerophyllous vegetation,10,0.344755,0.000000,intermediate_support
85234,Scena_18,7917,321,"Natural grassland, pasture, and fallow land",25,0.344920,0.040000,intermediate_support



spectrally_typical


,scene_id,field_id,class_label,class_name,n_pixels,spectral_support_median,outlier_fraction,audit_flag_recomputed
182489,Scena_8,14590,311,Broad-leaved forest,26,1.0,0.0000,spectrally_typical
224,Scena_1,368,223,Olive groves,15,1.0,0.0000,spectrally_typical
223,Scena_1,367,223,Olive groves,12,1.0,0.0000,spectrally_typical
222,Scena_1,366,223,Olive groves,27,1.0,0.0000,spectrally_typical
221,Scena_1,365,223,Olive groves,18,1.0,0.0000,spectrally_typical
220,Scena_1,364,223,Olive groves,160,1.0,0.0125,spectrally_typical
217,Scena_1,361,223,Olive groves,125,1.0,0.0000,spectrally_typical
214,Scena_1,358,223,Olive groves,95,1.0,0.0000,spectrally_typical
213,Scena_1,357,223,Olive groves,36,1.0,0.0000,spectrally_typical
212,Scena_1,356,223,Olive groves,41,1.0,0.0000,spectrally_typical


In [18]:
field_main["sa_global"] = field_main["audit_flag_recomputed"].eq("spectrally_atypical")

print("Main SA:", int(field_main["sa_global"].sum()))

Main SA: 5805


## 7. Leave-one-scene-out prototype scoring

For each held-out scene, prototypes and robust distance statistics are estimated from all other scenes. Only the held-out scene is then scored using these leave-one-scene-out quantities.

In [7]:
def compute_loso_pixel_scores():
    rng_seed_base = int(LOSO_RANDOM_SEED)
    parts = []
    proto_diag_rows = []

    for holdout_scene in all_scenes:
        print(f"\n=== Hold-out scene: {holdout_scene} ===")

        for class_label in supported_classes:
            test_idx = np.where((scene_ids == holdout_scene) & (y == class_label))[0]
            if len(test_idx) == 0:
                continue

            train_idx_all = np.where((scene_ids != holdout_scene) & (y == class_label))[0]
            if len(train_idx_all) == 0:
                print(f"  class {class_label}: no training pixels outside held-out scene; skipped")
                continue

            train_fields = pd.DataFrame({
                "scene_id": scene_ids[train_idx_all],
                "field_id": field_ids[train_idx_all],
            }).drop_duplicates()
            train_n_pixels = len(train_idx_all)
            train_n_fields = len(train_fields)
            train_n_scenes = pd.Series(scene_ids[train_idx_all]).nunique()

            train_supported = (
                (train_n_pixels >= MIN_CLASS_PIXELS) and
                (train_n_fields >= MIN_CLASS_FIELDS) and
                (train_n_scenes >= max(1, MIN_CLASS_SCENES - 1))
            )
            if not train_supported:
                print(
                    f"  class {class_label}: insufficient LOSO support "
                    f"(pixels={train_n_pixels}, fields={train_n_fields}, scenes={train_n_scenes}); skipped"
                )
                proto_diag_rows.append({
                    "holdout_scene": holdout_scene,
                    "class_label": int(class_label),
                    "train_n_pixels": int(train_n_pixels),
                    "train_n_fields": int(train_n_fields),
                    "train_n_scenes": int(train_n_scenes),
                    "train_supported": False,
                    "test_n_pixels": int(len(test_idx)),
                })
                continue

            seed = rng_seed_base + int(class_label) + 1000 * (all_scenes.index(holdout_scene) + 1)
            train_idx = maybe_subsample_indices(train_idx_all, max_n=LOSO_MAX_TRAIN_PIXELS_PER_CLASS, seed=seed)

            print(
                f"  class {class_label}: train={len(train_idx)} pixels "
                f"({train_n_scenes} scenes), test={len(test_idx)} pixels"
            )

            X_train = X[train_idx].astype(np.float32, copy=False)
            proto = estimate_median_proto(X_train)

            # Robust distance statistics are estimated on training pixels only.
            sad_train, ed_train = spectral_scores_to_proto(X_train, proto, chunk_size=CHUNK_SIZE)
            sad_q25, sad_med, sad_q75, sad_iqr = robust_median_iqr(sad_train)
            ed_q25, ed_med, ed_q75, ed_iqr = robust_median_iqr(ed_train)

            sad_thr = sad_q75 + 1.5 * sad_iqr
            ed_thr = ed_q75 + 1.5 * ed_iqr

            X_test = X[test_idx].astype(np.float32, copy=False)
            sad_test, ed_test = spectral_scores_to_proto(X_test, proto, chunk_size=CHUNK_SIZE)
            support, sad_z, ed_z = support_from_distances(sad_test, ed_test, sad_med, sad_iqr, ed_med, ed_iqr)
            is_outlier = (sad_test > sad_thr) | (ed_test > ed_thr)

            part = pd.DataFrame({
                "sample_index": sample_index[test_idx],
                "scene_id": scene_ids[test_idx],
                "field_id": field_ids[test_idx],
                "class_label": y[test_idx],
                "holdout_scene": holdout_scene,
                "sad": sad_test.astype(np.float32),
                "edist": ed_test.astype(np.float32),
                "sad_robust_z": sad_z.astype(np.float32),
                "edist_robust_z": ed_z.astype(np.float32),
                "spectral_support": support.astype(np.float32),
                "is_outlier": is_outlier.astype(bool),
            })
            parts.append(part)

            proto_diag_rows.append({
                "holdout_scene": holdout_scene,
                "class_label": int(class_label),
                "train_n_pixels": int(train_n_pixels),
                "train_n_fields": int(train_n_fields),
                "train_n_scenes": int(train_n_scenes),
                "train_supported": True,
                "test_n_pixels": int(len(test_idx)),
                "train_n_pixels_used": int(len(train_idx)),
                "sad_median_train": float(sad_med),
                "sad_iqr_train": float(sad_iqr),
                "edist_median_train": float(ed_med),
                "edist_iqr_train": float(ed_iqr),
            })

            del X_train, X_test, sad_train, ed_train, sad_test, ed_test, support, sad_z, ed_z, is_outlier, part
            gc.collect()

    if not parts:
        raise ValueError("No LOSO pixel scores were computed.")

    pixel_loso = pd.concat(parts, ignore_index=True)
    proto_diag = pd.DataFrame(proto_diag_rows)
    return pixel_loso, proto_diag

pixel_loso, loso_proto_diagnostics = compute_loso_pixel_scores()
print("pixel_loso:", pixel_loso.shape)
display(pixel_loso.head())
display(loso_proto_diagnostics.head())



=== Hold-out scene: Scena_1 ===
  class 221: train=241496 pixels (23 scenes), test=3308 pixels
  class 222: train=113739 pixels (24 scenes), test=4070 pixels
  class 223: train=706547 pixels (24 scenes), test=27566 pixels
  class 231: train=3626 pixels (16 scenes), test=67 pixels
  class 241: train=22723 pixels (23 scenes), test=2487 pixels
  class 242: train=3685 pixels (23 scenes), test=100 pixels
  class 243: train=1718 pixels (20 scenes), test=1 pixels
  class 311: train=122219 pixels (24 scenes), test=16176 pixels
  class 312: train=27457 pixels (23 scenes), test=930 pixels
  class 313: train=23731 pixels (22 scenes), test=185 pixels
  class 314: train=16875 pixels (23 scenes), test=494 pixels
  class 321: train=139970 pixels (24 scenes), test=2673 pixels
  class 322: train=32285 pixels (24 scenes), test=1040 pixels
  class 323: train=62402 pixels (22 scenes), test=4639 pixels
  class 331: train=1935 pixels (18 scenes), test=6 pixels
  class 333: train=6487 pixels (22 scenes), te

,sample_index,scene_id,field_id,class_label,holdout_scene,sad,edist,sad_robust_z,edist_robust_z,spectral_support,is_outlier
0,3,Scena_1,3045,221,Scena_1,0.088269,0.088241,-0.540227,-0.540951,1.000000,False
1,9,Scena_1,6127,221,Scena_1,0.081480,0.081456,-0.601872,-0.602756,1.000000,False
2,122,Scena_1,3127,221,Scena_1,0.097091,0.097054,-0.460116,-0.460673,1.000000,False
3,125,Scena_1,3148,221,Scena_1,0.085083,0.085058,-0.569150,-0.569948,1.000000,False
4,161,Scena_1,9081,221,Scena_1,0.148066,0.147931,0.002773,0.002771,0.997231,False


,holdout_scene,class_label,train_n_pixels,train_n_fields,train_n_scenes,train_supported,test_n_pixels,train_n_pixels_used,sad_median_train,sad_iqr_train,edist_median_train,edist_iqr_train
0,Scena_1,221,241496,26598,23,True,3308,241496.0,0.147761,0.110124,0.147627,0.109780
1,Scena_1,222,113739,12204,24,True,4070,113739.0,0.123075,0.131115,0.122998,0.130738
2,Scena_1,223,706547,42122,24,True,27566,706547.0,0.115380,0.110228,0.115316,0.109982
3,Scena_1,231,3626,592,16,True,67,3626.0,0.137182,0.120240,0.137075,0.119887
4,Scena_1,241,22723,4595,23,True,2487,22723.0,0.092917,0.080720,0.092883,0.080601


In [24]:
# Find candidate LOSO pixel-level dataframes safely
for name, obj in list(globals().items()):
    if hasattr(obj, "shape") and hasattr(obj, "columns"):
        cols = set(obj.columns)
        if (
            {"scene_id", "field_id", "class_label"}.issubset(cols)
            and any(c in cols for c in ["spectral_support", "support_score", "reliability"])
        ):
            print("\nCandidate:", name)
            print("shape:", obj.shape)
            print("columns:", list(obj.columns))


Candidate: pixel_loso
shape: (2497940, 11)
columns: ['sample_index', 'scene_id', 'field_id', 'class_label', 'holdout_scene', 'sad', 'edist', 'sad_robust_z', 'edist_robust_z', 'spectral_support', 'is_outlier']


In [25]:
[x for x in globals().keys() if "loso" in x.lower()]

['LOSO_MAX_TRAIN_PIXELS_PER_CLASS',
 'LOSO_RANDOM_SEED',
 'compute_loso_pixel_scores',
 'pixel_loso',
 'loso_proto_diagnostics',
 'field_loso',
 'loso_thresholds',
 'field_loso_path',
 'loso_diag_path',
 'loso_cols',
 'missing_loso',
 'loso_sa',
 'loso_sa_recall',
 'loso_sa_rate',
 'eligible_loso',
 'tau10_loso',
 'tau30_loso',
 'omega90_loso',
 'loso_cmp',
 'loso_total',
 'required_loso_cols']

In [19]:
# =========================================================
# RECOMPUTE LOSO SPECTRAL-SUPPORT STRATA
# =========================================================

eligible_loso = (
    field_loso["class_supported"].eq(True)
    & (field_loso["n_pixels"] >= MIN_FIELD_PIXELS)
    & field_loso[score_col].notna()
    & field_loso[out_col].notna()
)

tau10_loso = field_loso.loc[eligible_loso, score_col].quantile(0.10)
tau30_loso = field_loso.loc[eligible_loso, score_col].quantile(0.30)
omega90_loso = field_loso.loc[eligible_loso, out_col].quantile(0.90)

print("Eligible LOSO:", int(eligible_loso.sum()))
print("tau10_loso:", tau10_loso)
print("tau30_loso:", tau30_loso)
print("omega90_loso:", omega90_loso)

field_loso["audit_flag_recomputed"] = field_loso["audit_flag"].copy()

field_loso.loc[eligible_loso, "audit_flag_recomputed"] = "spectrally_typical"

field_loso.loc[
    eligible_loso
    & (field_loso[score_col] > tau10_loso)
    & (field_loso[score_col] <= tau30_loso)
    & (field_loso[out_col] < omega90_loso),
    "audit_flag_recomputed"
] = "intermediate_support"

field_loso.loc[
    eligible_loso
    & (
        (field_loso[score_col] <= tau10_loso)
        | (field_loso[out_col] >= omega90_loso)
    ),
    "audit_flag_recomputed"
] = "spectrally_atypical"

print(field_loso["audit_flag_recomputed"].value_counts(dropna=False))

field_loso["sa_loso"] = field_loso["audit_flag_recomputed"].eq("spectrally_atypical")

print("LOSO SA:", int(field_loso["sa_loso"].sum()))

Eligible LOSO: 40792
tau10_loso: 0.289933842420578
tau30_loso: 0.6866386950016021
omega90_loso: 0.1
audit_flag_recomputed
too_small                     150076
spectrally_typical             28101
intermediate_support            7162
spectrally_atypical             5529
insufficient_class_support        38
unassigned                         8
Name: count, dtype: int64
LOSO SA: 5529


In [20]:
main_cmp = field_main.loc[
    eligible_main,
    ["scene_id", "field_id", "class_label", score_col, out_col, "sa_global"]
].copy()

loso_cmp = field_loso.loc[
    eligible_loso,
    ["scene_id", "field_id", score_col, out_col, "sa_loso"]
].copy()

main_cmp["scene_id"] = main_cmp["scene_id"].astype(str)
main_cmp["field_id"] = main_cmp["field_id"].astype(str)

loso_cmp["scene_id"] = loso_cmp["scene_id"].astype(str)
loso_cmp["field_id"] = loso_cmp["field_id"].astype(str)

merged = main_cmp.merge(
    loso_cmp,
    on=["scene_id", "field_id"],
    how="inner",
    suffixes=("_global", "_loso")
)

print("Merged:", merged.shape)
print("Main SA:", int(merged["sa_global"].sum()))
print("LOSO SA:", int(merged["sa_loso"].sum()))

Merged: (40792, 9)
Main SA: 5805
LOSO SA: 5529


In [21]:
import numpy as np
from scipy.stats import spearmanr

main_sa = merged["sa_global"].astype(bool)
loso_sa = merged["sa_loso"].astype(bool)

intersection = int((main_sa & loso_sa).sum())
union = int((main_sa | loso_sa).sum())
main_total = int(main_sa.sum())
loso_total = int(loso_sa.sum())

jaccard = intersection / union if union > 0 else np.nan
recall_main = intersection / main_total if main_total > 0 else np.nan

rho_score = spearmanr(
    merged[f"{score_col}_global"],
    merged[f"{score_col}_loso"],
    nan_policy="omit"
).statistic

print("rho_score:", rho_score)
print("Main SA:", main_total)
print("LOSO SA:", loso_total)
print("Intersection:", intersection)
print("Union:", union)
print("Jaccard:", jaccard)
print("Recall main:", recall_main)

rho_score: 0.9793424718458621
Main SA: 5805
LOSO SA: 5529
Intersection: 4965
Union: 6369
Jaccard: 0.7795572303344324
Recall main: 0.8552971576227391


In [ ]:
print(pixel_loso.shape)
print(pixel_loso.columns.tolist())
display(pixel_loso.head())

## 8. Aggregate LOSO scores to field level

In [26]:
# =========================================================
# 8. Aggregate LOSO scores to field level and assign strata
# =========================================================

print("Aggregating LOSO pixel-level scores to field-scene level...")

loso_scores_df = pixel_loso.copy()

print("LOSO pixel table:", loso_scores_df.shape)
print(loso_scores_df.columns.tolist())

# Expected input:
# loso_scores_df must contain one row per valid LOSO-scored pixel
# with at least:
# scene_id, field_id, class_label, spectral_support, sad, edist, is_outlier
#
# If your notebook uses a different variable name for the LOSO pixel table,
# replace loso_scores_df below with that name.

required_loso_cols = [
    "scene_id",
    "field_id",
    "class_label",
    "spectral_support",
    "sad",
    "edist",
    "is_outlier",
]

missing = [c for c in required_loso_cols if c not in loso_scores_df.columns]
if missing:
    raise ValueError(f"Missing required LOSO score columns: {missing}")

# ---------------------------------------------------------
# Aggregate LOSO scores by field-scene unit
# ---------------------------------------------------------

field_loso = (
    loso_scores_df
    .groupby(["scene_id", "field_id", "class_label"], as_index=False)
    .agg(
        n_pixels=("spectral_support", "size"),
        spectral_support_mean=("spectral_support", "mean"),
        spectral_support_median=("spectral_support", "median"),
        spectral_support_q25=("spectral_support", lambda x: x.quantile(0.25)),
        spectral_support_q75=("spectral_support", lambda x: x.quantile(0.75)),
        sad_median=("sad", "median"),
        sad_mean=("sad", "mean"),
        sad_q25=("sad", lambda x: x.quantile(0.25)),
        sad_q75=("sad", lambda x: x.quantile(0.75)),
        edist_median=("edist", "median"),
        edist_mean=("edist", "mean"),
        outlier_fraction=("is_outlier", "mean"),
    )
)

# ---------------------------------------------------------
# Add class metadata if available
# ---------------------------------------------------------

if "class_info_df" in globals():
    meta_cols = [
        c for c in [
            "class_label",
            "class_name",
            "persistence_group",
            "class_n_pixels",
            "class_n_fields",
            "class_n_scenes",
            "class_supported",
        ]
        if c in class_info_df.columns
    ]

    field_loso = field_loso.merge(
        class_info_df[meta_cols].drop_duplicates("class_label"),
        on="class_label",
        how="left"
    )

else:
    print("Warning: class_info_df not found. Creating minimal class support metadata.")
    field_loso["class_supported"] = True

# Ensure class_supported exists
if "class_supported" not in field_loso.columns:
    field_loso["class_supported"] = True

# ---------------------------------------------------------
# Initial audit flag: too_small / insufficient support / placeholder typical
# ---------------------------------------------------------

field_loso["audit_flag"] = "spectrally_typical"

field_loso.loc[
    field_loso["class_supported"].ne(True),
    "audit_flag"
] = "insufficient_class_support"

field_loso.loc[
    field_loso["n_pixels"] < MIN_FIELD_PIXELS,
    "audit_flag"
] = "too_small"

# ---------------------------------------------------------
# Recompute LOSO spectral-support strata on eligible fields
# ---------------------------------------------------------

score_col = "spectral_support_median"
out_col = "outlier_fraction"

eligible_loso = (
    field_loso["class_supported"].eq(True)
    & (field_loso["n_pixels"] >= MIN_FIELD_PIXELS)
    & field_loso[score_col].notna()
    & field_loso[out_col].notna()
)

tau10_loso = field_loso.loc[eligible_loso, score_col].quantile(0.10)
tau30_loso = field_loso.loc[eligible_loso, score_col].quantile(0.30)
omega90_loso = field_loso.loc[eligible_loso, out_col].quantile(0.90)

print("Eligible LOSO field-scene units:", int(eligible_loso.sum()))
print("tau10_loso:", tau10_loso)
print("tau30_loso:", tau30_loso)
print("omega90_loso:", omega90_loso)

field_loso["audit_flag_recomputed"] = field_loso["audit_flag"].copy()

# Default eligible units to ST
field_loso.loc[
    eligible_loso,
    "audit_flag_recomputed"
] = "spectrally_typical"

# Intermediate support
field_loso.loc[
    eligible_loso
    & (field_loso[score_col] > tau10_loso)
    & (field_loso[score_col] <= tau30_loso)
    & (field_loso[out_col] < omega90_loso),
    "audit_flag_recomputed"
] = "intermediate_support"

# Spectrally atypical
field_loso.loc[
    eligible_loso
    & (
        (field_loso[score_col] <= tau10_loso)
        | (field_loso[out_col] >= omega90_loso)
    ),
    "audit_flag_recomputed"
] = "spectrally_atypical"

field_loso["sa_loso"] = field_loso["audit_flag_recomputed"].eq("spectrally_atypical")

print("\nLOSO audit flags:")
print(field_loso["audit_flag_recomputed"].value_counts(dropna=False))

print("\nLOSO SA:", int(field_loso["sa_loso"].sum()))

# ---------------------------------------------------------
# Save LOSO field-level table
# ---------------------------------------------------------

field_loso_path = TABLE_DIR / "field_spectral_support_loso_prototypes_full.csv"
field_loso.to_csv(field_loso_path, index=False)

print("\nSaved:", field_loso_path)

Aggregating LOSO pixel-level scores to field-scene level...
LOSO pixel table: (2497940, 11)
['sample_index', 'scene_id', 'field_id', 'class_label', 'holdout_scene', 'sad', 'edist', 'sad_robust_z', 'edist_robust_z', 'spectral_support', 'is_outlier']
Eligible LOSO field-scene units: 40792
tau10_loso: 0.289933842420578
tau30_loso: 0.6866386950016021
omega90_loso: 0.1

LOSO audit flags:
audit_flag_recomputed
too_small               149752
spectrally_typical       28101
intermediate_support      7162
spectrally_atypical       5529
Name: count, dtype: int64

LOSO SA: 5529

Saved: /kaggle/working/tables/field_spectral_support_loso_prototypes_full.csv


## 9. Compare global and leave-one-scene-out configurations

In [27]:

# =========================================================
# 9. Compare global and leave-one-scene-out configurations
# =========================================================

from scipy.stats import spearmanr
import numpy as np
import pandas as pd

print("Comparing global-prototype and LOSO-prototype configurations...")

score_col = "spectral_support_median"
out_col = "outlier_fraction"

# ---------------------------------------------------------
# Ensure main/global strata exist
# ---------------------------------------------------------

if "sa_global" not in field_main.columns:
    print("sa_global not found. Recomputing main/global strata...")

    eligible_main = (
        field_main["class_supported"].eq(True)
        & (field_main["n_pixels"] >= MIN_FIELD_PIXELS)
        & field_main[score_col].notna()
        & field_main[out_col].notna()
    )

    tau10_main = field_main.loc[eligible_main, score_col].quantile(0.10)
    tau30_main = field_main.loc[eligible_main, score_col].quantile(0.30)
    omega90_main = field_main.loc[eligible_main, out_col].quantile(0.90)

    field_main["audit_flag_recomputed"] = field_main["audit_flag"].copy()

    field_main.loc[
        eligible_main,
        "audit_flag_recomputed"
    ] = "spectrally_typical"

    field_main.loc[
        eligible_main
        & (field_main[score_col] > tau10_main)
        & (field_main[score_col] <= tau30_main)
        & (field_main[out_col] < omega90_main),
        "audit_flag_recomputed"
    ] = "intermediate_support"

    field_main.loc[
        eligible_main
        & (
            (field_main[score_col] <= tau10_main)
            | (field_main[out_col] >= omega90_main)
        ),
        "audit_flag_recomputed"
    ] = "spectrally_atypical"

    field_main["sa_global"] = field_main["audit_flag_recomputed"].eq("spectrally_atypical")

else:
    eligible_main = (
        field_main["class_supported"].eq(True)
        & (field_main["n_pixels"] >= MIN_FIELD_PIXELS)
        & field_main[score_col].notna()
        & field_main[out_col].notna()
    )

# ---------------------------------------------------------
# Ensure LOSO strata exist
# ---------------------------------------------------------

if "sa_loso" not in field_loso.columns:
    raise ValueError(
        "sa_loso not found in field_loso. Run Section 8 first."
    )

eligible_loso = (
    field_loso["class_supported"].eq(True)
    & (field_loso["n_pixels"] >= MIN_FIELD_PIXELS)
    & field_loso[score_col].notna()
    & field_loso[out_col].notna()
)

# ---------------------------------------------------------
# Prepare comparison tables
# ---------------------------------------------------------

main_cmp = field_main.loc[
    eligible_main,
    ["scene_id", "field_id", "class_label", score_col, out_col, "sa_global"]
].copy()

loso_cmp = field_loso.loc[
    eligible_loso,
    ["scene_id", "field_id", score_col, out_col, "sa_loso"]
].copy()

# Normalize merge keys
for df in [main_cmp, loso_cmp]:
    df["scene_id"] = df["scene_id"].astype(str)
    df["field_id"] = df["field_id"].astype(str)

merged = main_cmp.merge(
    loso_cmp,
    on=["scene_id", "field_id"],
    how="inner",
    suffixes=("_global", "_loso")
)

print("Main eligible:", len(main_cmp))
print("LOSO eligible:", len(loso_cmp))
print("Merged eligible:", len(merged))

if len(merged) == 0:
    raise ValueError("Global and LOSO field tables did not merge. Check scene_id and field_id types/values.")

# ---------------------------------------------------------
# Field-level score correlation
# ---------------------------------------------------------

rho_score = spearmanr(
    merged[f"{score_col}_global"],
    merged[f"{score_col}_loso"],
    nan_policy="omit"
).statistic

# ---------------------------------------------------------
# SA overlap metrics
# ---------------------------------------------------------

main_sa = merged["sa_global"].astype(bool)
loso_sa = merged["sa_loso"].astype(bool)

intersection = int((main_sa & loso_sa).sum())
union = int((main_sa | loso_sa).sum())

main_sa_total = int(main_sa.sum())
loso_sa_total = int(loso_sa.sum())

jaccard = intersection / union if union > 0 else np.nan
main_recall = intersection / main_sa_total if main_sa_total > 0 else np.nan
loso_recall = intersection / loso_sa_total if loso_sa_total > 0 else np.nan

sa_diff = loso_sa_total - main_sa_total

# ---------------------------------------------------------
# Scene-level SA-rate correlation
# ---------------------------------------------------------

scene_rates = (
    merged
    .groupby("scene_id", as_index=False)
    .agg(
        n_fields=("field_id", "size"),
        sa_rate_global=("sa_global", "mean"),
        sa_rate_loso=("sa_loso", "mean"),
        sa_global=("sa_global", "sum"),
        sa_loso=("sa_loso", "sum"),
    )
)

if scene_rates["sa_rate_global"].nunique() > 1 and scene_rates["sa_rate_loso"].nunique() > 1:
    rho_scene = spearmanr(
        scene_rates["sa_rate_global"],
        scene_rates["sa_rate_loso"],
        nan_policy="omit"
    ).statistic
else:
    rho_scene = np.nan

# ---------------------------------------------------------
# Summary table
# ---------------------------------------------------------

summary = pd.DataFrame([
    {
        "prototype_setting": "Global prototypes",
        "rho_field_support": 1.000,
        "sa": main_sa_total,
        "jaccard_sa": 1.000,
        "main_sa_recall": 1.000,
        "loso_sa_recall": 1.000,
        "sa_diff": 0,
        "rho_scene_sa_rate": 1.000,
    },
    {
        "prototype_setting": "Leave-one-scene-out",
        "rho_field_support": rho_score,
        "sa": loso_sa_total,
        "jaccard_sa": jaccard,
        "main_sa_recall": main_recall,
        "loso_sa_recall": loso_recall,
        "sa_diff": sa_diff,
        "rho_scene_sa_rate": rho_scene,
    },
])

print("\nSummary:")
display(summary)

print("\nScene-level comparison:")
display(scene_rates.head())

# ---------------------------------------------------------
# Save outputs
# ---------------------------------------------------------

comparison_path = TABLE_DIR / "field_loso_vs_global_comparison_full.csv"
scene_path = TABLE_DIR / "scene_loso_prototype_sensitivity_full.csv"
summary_path = TABLE_DIR / "loso_prototype_sensitivity_summary_full.csv"

merged.to_csv(comparison_path, index=False)
scene_rates.to_csv(scene_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nSaved:")
print(comparison_path)
print(scene_path)
print(summary_path)

# ---------------------------------------------------------
# Export LaTeX table
# ---------------------------------------------------------

latex_table = f"""
\\begin{{table}}[t]
\\centering
\\scriptsize
\\caption{{Sensitivity of spectral-support strata to leave-one-scene-out
prototype estimation.}}
\\label{{tab:sensitivity_loso_prototypes}}
\\begin{{tabular}}{{lrrrrr}}
\\toprule
Prototype setting & $\\rho(S_f)$ & SA & Jac. & Main rec. & SA diff. \\\\
\\midrule
Global prototypes & 1.000 & {main_sa_total:,} & 100.0 & 100.0 & 0 \\\\
Leave-one-scene-out & {rho_score:.3f} & {loso_sa_total:,} & {100*jaccard:.1f} & {100*main_recall:.1f} & {sa_diff:+d} \\\\
\\bottomrule
\\end{{tabular}}

\\vspace{{1mm}}
\\footnotesize
$\\rho(S_f)$ is the Spearman rank correlation of field-level spectral-support
scores with the main global-prototype configuration. Jac. is the Jaccard
overlap of the spectrally atypical sets. Main rec. is the recall of the main
spectrally atypical set. SA diff. is the difference in the number of
spectrally atypical field--scene units relative to the main configuration.
\\end{{table}}
""".strip()

latex_path = TABLE_DIR / "tab_sensitivity_loso_prototypes_full.tex"
latex_path.write_text(latex_table)

print("\nLaTeX table:")
print(latex_table)
print("\nSaved:", latex_path)

Comparing global-prototype and LOSO-prototype configurations...
Main eligible: 40800
LOSO eligible: 40792
Merged eligible: 40792

Summary:


,prototype_setting,rho_field_support,sa,jaccard_sa,main_sa_recall,loso_sa_recall,sa_diff,rho_scene_sa_rate
0,Global prototypes,1.000000,5805,1.000000,1.000000,1.000000,0,1.000000
1,Leave-one-scene-out,0.979342,5529,0.779557,0.855297,0.897992,-276,0.989231



Scene-level comparison:


,scene_id,n_fields,sa_rate_global,sa_rate_loso,sa_global,sa_loso
0,Scena_1,1991,0.047212,0.031140,94,62
1,Scena_10,1650,0.133939,0.135758,221,224
2,Scena_11,1922,0.238293,0.204475,458,393
3,Scena_12,1946,0.018499,0.014388,36,28
4,Scena_13,2174,0.040938,0.025759,89,56



Saved:
/kaggle/working/tables/field_loso_vs_global_comparison_full.csv
/kaggle/working/tables/scene_loso_prototype_sensitivity_full.csv
/kaggle/working/tables/loso_prototype_sensitivity_summary_full.csv

LaTeX table:
\begin{table}[t]
\centering
\scriptsize
\caption{Sensitivity of spectral-support strata to leave-one-scene-out
prototype estimation.}
\label{tab:sensitivity_loso_prototypes}
\begin{tabular}{lrrrrr}
\toprule
Prototype setting & $\rho(S_f)$ & SA & Jac. & Main rec. & SA diff. \\
\midrule
Global prototypes & 1.000 & 5,805 & 100.0 & 100.0 & 0 \\
Leave-one-scene-out & 0.979 & 5,529 & 78.0 & 85.5 & -276 \\
\bottomrule
\end{tabular}

\vspace{1mm}
\footnotesize
$\rho(S_f)$ is the Spearman rank correlation of field-level spectral-support
scores with the main global-prototype configuration. Jac. is the Jaccard
overlap of the spectrally atypical sets. Main rec. is the recall of the main
spectrally atypical set. SA diff. is the difference in the number of
spectrally atypical field--sc

## 10. LaTeX table for the manuscript appendix

In [28]:
row = summary.iloc[0]

tex = rf"""
\begin{{table}}[t]
\centering
\scriptsize
\caption{{Sensitivity of the spectral-support audit to leave-one-scene-out prototype estimation.}}
\label{{tab:sensitivity_loso_prototypes}}
\begin{{tabular}}{{lrrrrrr}}
\toprule
Prototype setting & $\rho(S_f)$ & Jac. & Rec. & SA main & SA LOSO & $\rho_{{\mathrm{{scene}}}}$ \\
\midrule
Global prototypes & 1.000 & 100.0 & 100.0 & {int(row['main_n_sa']):,} & {int(row['main_n_sa']):,} & 1.00 \\
Leave-one-scene-out & {row['rho_support']:.3f} & {100*row['jaccard_sa']:.1f} & {100*row['main_sa_recall']:.1f} & {int(row['main_n_sa']):,} & {int(row['loso_n_sa']):,} & {row['rho_scene_sa_rate']:.2f} \\
\bottomrule
\end{{tabular}}

\vspace{{1mm}}
\footnotesize
Jac. = Jaccard overlap of the spectrally-atypical field--scene units relative to the main global-prototype configuration; Rec. = recall of the main spectrally-atypical set; $\rho(S_f)$ = Spearman rank correlation of field-level spectral-support scores; $\rho_{{\mathrm{{scene}}}}$ = Spearman correlation of scene-level spectrally-atypical rates.
\end{{table}}
""".strip()

tex_path = TABLE_DIR / f"tab_sensitivity_loso_prototypes_{SETTING}.tex"
tex_path.write_text(tex)

print(tex)
print("Saved:", tex_path)


KeyError: 'main_n_sa'

## 11. Suggested text for the appendix

Use this text only after checking that the numerical results are stable.

In [11]:
text = rf"""
\subsection{{Sensitivity to leave-one-scene-out prototype estimation}}
\label{{app:sensitivity_loso_prototypes}}

Because the class prototypes were estimated from the same legacy SIT labels that were subsequently audited, we evaluated whether the results were sensitive to scene-level contributions to the prototype estimates. In a leave-one-scene-out configuration, each PRISMA scene was scored using class prototypes estimated from all remaining scenes. Robust class-wise distance statistics were also estimated from the remaining scenes, and the held-out scene was then scored using the same spectral-support formulation as in the main analysis.

The leave-one-scene-out configuration was compared with the main global-prototype configuration using the Spearman rank correlation of field-level support scores, the Jaccard overlap of the spectrally-atypical set, the recall of the main spectrally-atypical set, and the Spearman correlation of scene-level spectrally-atypical rates. The purpose of this analysis was not to provide an independent semantic validation, but to assess whether the audit was dominated by a single scene or by scene-specific prototype contamination.

\input{{tables/tab_sensitivity_loso_prototypes_{SETTING}}}
""".strip()

print(text)


\subsection{Sensitivity to leave-one-scene-out prototype estimation}
\label{app:sensitivity_loso_prototypes}

Because the class prototypes were estimated from the same legacy SIT labels that were subsequently audited, we evaluated whether the results were sensitive to scene-level contributions to the prototype estimates. In a leave-one-scene-out configuration, each PRISMA scene was scored using class prototypes estimated from all remaining scenes. Robust class-wise distance statistics were also estimated from the remaining scenes, and the held-out scene was then scored using the same spectral-support formulation as in the main analysis.

The leave-one-scene-out configuration was compared with the main global-prototype configuration using the Spearman rank correlation of field-level support scores, the Jaccard overlap of the spectrally-atypical set, the recall of the main spectrally-atypical set, and the Spearman correlation of scene-level spectrally-atypical rates. The purpose of this 